① 반입물량 JSON 생성 → ② 반입물량에서 2022년 1월 품목 목록 확보 → ③ 그 품목 목록으로 경매결과 API 호출 → ④ 경매결과 JSON 생성 → ⑤ 그다음 02_generated_data_check.ipynb에서 분포 분석

In [8]:
#%pip install requests
#%pip install dotenv

In [9]:
from pathlib import Path
from datetime import datetime, timedelta
import os
import json
import time
import requests
import pandas as pd

from dotenv import load_dotenv


# 프로젝트 루트
if Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()

load_dotenv(PROJECT_ROOT / ".env")

GARAK_AUCTION_PASSWORD = os.getenv("GARAK_AUCTION_PASSWORD")
GARAK_ARRIVAL_PASSWORD = os.getenv("GARAK_ARRIVAL_PASSWORD")

BASE_URL = "http://www.garak.co.kr/homepage/publicdata/dataJsonOpen.do"

ARRIVAL_DIR = PROJECT_ROOT / "data" / "raw" / "garak" / "arrival"
AUCTION_DIR = PROJECT_ROOT / "data" / "raw" / "garak" / "auction"

ARRIVAL_DIR.mkdir(parents=True, exist_ok=True)
AUCTION_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("ARRIVAL_DIR:", ARRIVAL_DIR)
print("AUCTION_DIR:", AUCTION_DIR)

PROJECT_ROOT: /Users/yoon/Documents/code/5PL
ARRIVAL_DIR: /Users/yoon/Documents/code/5PL/data/raw/garak/arrival
AUCTION_DIR: /Users/yoon/Documents/code/5PL/data/raw/garak/auction


In [10]:
def fetch_arrival_page(date, page=1, page_size=100):
    params = {
        "id": "11115",
        "passwd": GARAK_ARRIVAL_PASSWORD,
        "dataid": "data20",
        "pagesize": page_size,
        "pageidx": page,
        "portal.templet": "false",
        "date": date,
    }

    response = requests.get(
        BASE_URL,
        params=params,
        timeout=30
    )
    response.raise_for_status()

    try:
        data = response.json()
    except Exception:
        print("JSON 응답 아님:", date)
        print(response.text[:300])
        return []

    return data.get("resultData", [])

In [11]:
def fetch_all_arrival(date, page_size=100):
    all_records = []

    for page in range(1, 100):
        records = fetch_arrival_page(
            date=date,
            page=page,
            page_size=page_size
        )

        if not records:
            break

        all_records.extend(records)

        if len(records) < page_size:
            break

        time.sleep(0.2)

    return all_records

In [12]:
date = "20220103"

records = fetch_all_arrival(date)

print("행 수:", len(records))

if records:
    save_path = ARRIVAL_DIR / f"arrival_{date}.json"

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(
            {
                "date": date,
                "resultData": records
            },
            f,
            ensure_ascii=False,
            indent=2
        )

    print("저장 완료:", save_path)

행 수: 5
저장 완료: /Users/yoon/Documents/code/5PL/data/raw/garak/arrival/arrival_20220103.json


In [13]:
with open(
    ARRIVAL_DIR / "arrival_20220103.json",
    "r",
    encoding="utf-8"
) as f:
    test_data = json.load(f)

arrival_test = pd.DataFrame(test_data["resultData"])

display(arrival_test.head(30))

,A1,A2,BURYU,A3,ROWNO,A4,A5,A6,A7,TOT,구분
0,None,None,<전체>,None,1,None,None,None,4.8985,4.8985,합계
1,None,None,<전체>,None,2,None,None,None,4.8985,4.8985,기타채소
2,None,None,기타채소,None,3,None,None,None,0.1170,0.1170,숙주나물
3,None,None,기타채소,None,4,None,None,None,3.7815,3.7815,콩나물
4,None,None,기타채소,None,5,None,None,None,1.0000,1.0000,고사리


In [14]:
def make_date_list(start_date, end_date):
    start = datetime.strptime(start_date, "%Y%m%d")
    end = datetime.strptime(end_date, "%Y%m%d")

    dates = []

    while start <= end:
        dates.append(start.strftime("%Y%m%d"))
        start += timedelta(days=1)

    return dates

In [15]:
dates = make_date_list(
    "20220101",
    "20220131"
)

for date in dates:

    save_path = ARRIVAL_DIR / f"arrival_{date}.json"

    # 이미 있으면 재호출하지 않음
    if save_path.exists():
        print(date, "| 이미 존재")
        continue

    records = fetch_all_arrival(date)

    if not records:
        print(date, "| 데이터 없음")
        continue

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(
            {
                "date": date,
                "resultData": records
            },
            f,
            ensure_ascii=False,
            indent=2
        )

    print(date, "|", len(records), "행 저장")

    time.sleep(0.3)

20220101 | 111 행 저장
20220102 | 20 행 저장
20220103 | 이미 존재
20220104 | 116 행 저장
20220105 | 122 행 저장
20220106 | 117 행 저장
20220107 | 116 행 저장
20220108 | 115 행 저장
20220109 | 97 행 저장
20220110 | 46 행 저장
20220111 | 121 행 저장
20220112 | 126 행 저장
20220113 | 124 행 저장
20220114 | 126 행 저장
20220115 | 114 행 저장
20220116 | 97 행 저장
20220117 | 22 행 저장
20220118 | 109 행 저장
20220119 | 135 행 저장
20220120 | 116 행 저장
20220121 | 123 행 저장
20220122 | 119 행 저장
JSON 응답 아님: 20220123
<html lang='ko'>
<title>Data JSON Type</title>
<body>
{ LIST_COUNT: 0, ApiErrorMsg:데이터가 없습니다.}
</html>
</body>

20220123 | 100 행 저장
20220124 | 26 행 저장
20220125 | 128 행 저장
20220126 | 129 행 저장
20220127 | 135 행 저장
20220128 | 120 행 저장
20220129 | 124 행 저장
20220130 | 90 행 저장
20220131 | 59 행 저장


In [16]:
all_arrival_records = []

for file in sorted(ARRIVAL_DIR.glob("arrival_202201*.json")):

    with open(file, "r", encoding="utf-8") as f:
        data = json.load(f)

    all_arrival_records.extend(
        data.get("resultData", [])
    )

arrival_df = pd.DataFrame(all_arrival_records)

display(arrival_df.head())

,A1,A2,BURYU,A3,ROWNO,A4,A5,A6,A7,TOT,구분
0,None,None,<전체>,None,1,None,None,None,11.48340,11.48340,건어기타
1,None,None,<전체>,None,2,None,None,None,2.93900,2.93900,갑각류
2,None,None,<전체>,None,3,None,None,None,699.35965,699.35965,합계
3,None,None,<전체>,None,4,None,None,None,3.93020,3.93020,패류
4,None,None,<전체>,None,5,None,None,None,81.58200,81.58200,조미채류


In [17]:
item_rows = arrival_df[
    arrival_df["BURYU"] != "<전체>"
].copy()

In [18]:
items = sorted(
    item_rows["구분"]
    .dropna()
    .astype(str)
    .unique()
)

print("품목 수:", len(items))
print(items)

품목 수: 180
['가공 낙지', '가공 다랑어', '가공 문어', '가공 생선살류', '가공 생선알류', '가공 오징어', '가공 장어', '가공 조갯살류', '가공 주꾸미', '가오리', '가재', '갓', '개조개', '갯고둥', '건가지', '건고추', '건과류', '건말', '건새우', '건양미리', '건어류 기타', '건포도(청과)', '건한치', '건해삼', '게', '게지', '겨자잎', '고구마', '고구마순', '고둥', '고비', '고사리', '고수', '고추잎', '곤드레', '골뱅이', '곶감', '과일류 기타', '굴비', '근대', '기타 가공어류', '기타 갑각류', '기타 건어류', '기타 연체류', '기타 패류', '김', '꼬막', '꼴뚜기젓', '꽃게', '낙지', '냉이', '넙치(광어)', '노가리채', '논우렁', '농어', '달래', '당귀잎', '대추', '더덕', '도다리', '도라지', '돔류', '돗나물', '두릅', '두리안', '두부', '땅콩', '레몬', '로메인', '루꼴라', '마', '마늘', '말(청과)', '망고', '머위대', '멸치액젓', '멸치젓', '명란젓', '명태피포', '무말랭이', '무순', '묵', '물미역(청과)', '미꾸라지', '밤', '방울양배추', '방풍나물', '버섯', '베이비', '병어', '복어류', '볼락', '봄동배추', '봉지 미역', '북어', '북어채', '브로콜리', '블루베리', '비타민', '비트', '새싹', '새우젓', '생강', '석류', '세발나물', '셀러리', '속새', '수삼', '숙주나물', '신선초', '쌈추', '씀바귀', '아 몬 드', '아가미젓', '아귀', '아보카도', '아스파라거스', '아욱', '알파파', '야콘', '양상추', '어류 기타', '어리굴젓', '연근', '염장 다시마', '오징어', '오징어젓', '오징어채', '용과', '우엉', '원추리', '유채', '은행', '임연수어', '자몽', '잣', '

품목 선정

In [23]:
import json
import pandas as pd
from pathlib import Path

records = []

for file in sorted(ARRIVAL_DIR.glob("arrival_202201*.json")):
    date = file.stem.split("_")[-1]

    with open(file, "r", encoding="utf-8") as f:
        data = json.load(f)

    for r in data.get("resultData", []):
        # <전체> 행 제외: 실제 개별 품목만 사용
        if r.get("BURYU") == "<전체>":
            continue

        records.append({
            "date": date,
            "category": r.get("BURYU"),
            "item": r.get("구분"),
            "total_arrival": r.get("TOT")
        })

arrival_items = pd.DataFrame(records)

arrival_items["total_arrival"] = pd.to_numeric(
    arrival_items["total_arrival"],
    errors="coerce"
)

In [24]:
item_summary = (
    arrival_items
    .groupby("item")
    .agg(
        active_days=("date", "nunique"),
        total_arrival=("total_arrival", "sum"),
        mean_daily_arrival=("total_arrival", "mean"),
        median_daily_arrival=("total_arrival", "median"),
        std_daily_arrival=("total_arrival", "std")
    )
    .sort_values(
        ["active_days", "total_arrival"],
        ascending=False
    )
)

display(item_summary.head(50))

,active_days,total_arrival,mean_daily_arrival,median_daily_arrival,std_daily_arrival
item,,,,,
고사리,31,459.32600,14.816968,12.478000,11.801150
체리,30,246.20100,8.206700,8.207500,5.747163
마늘,29,2308.37100,79.599000,80.037000,33.385441
물미역(청과),29,1565.57800,53.985448,62.932000,21.503372
콩나물,29,1320.95900,45.550310,46.970600,16.903450
숙주나물,29,1013.76350,34.957362,27.684500,27.192542
파래(청과),29,349.25000,12.043103,13.680000,5.025101
톳(청과),29,261.17800,9.006138,9.892000,4.154255
망고,29,257.69100,8.885897,7.548000,6.065780


In [25]:
candidate_items = (
    item_summary[
        item_summary["active_days"] >= 15
    ]
    .head(20)
    .index
    .tolist()
)

candidate_items

['고사리',
 '체리',
 '마늘',
 '물미역(청과)',
 '콩나물',
 '숙주나물',
 '파래(청과)',
 '톳(청과)',
 '망고',
 '키위',
 '자몽',
 '아보카도',
 '블루베리',
 '토란대',
 '고비',
 '생강',
 '도라지',
 '연근',
 '석류',
 '레몬']

In [26]:
CANDIDATE_ITEMS = [
    "고사리", "체리", "마늘", "물미역(청과)", "콩나물",
    "숙주나물", "파래(청과)", "톳(청과)", "망고", "키위",
    "자몽", "아보카도", "블루베리", "토란대", "고비",
    "생강", "도라지", "연근", "석류", "레몬"
]

In [27]:
candidate_arrival = arrival_items[
    arrival_items["item"].isin(CANDIDATE_ITEMS)
].copy()

candidate_summary = (
    candidate_arrival
    .groupby("item")
    .agg(
        active_days=("date", "nunique"),
        total_arrival=("total_arrival", "sum"),
        mean_daily_arrival=("total_arrival", "mean"),
        median_daily_arrival=("total_arrival", "median"),
        std_daily_arrival=("total_arrival", "std"),
    )
)

candidate_summary["arrival_cv"] = (
    candidate_summary["std_daily_arrival"]
    / candidate_summary["mean_daily_arrival"]
)

candidate_summary = candidate_summary.sort_values(
    ["active_days", "total_arrival"],
    ascending=[False, False]
)

display(candidate_summary)

,active_days,total_arrival,mean_daily_arrival,median_daily_arrival,std_daily_arrival,arrival_cv
item,,,,,,
고사리,31,459.3260,14.816968,12.47800,11.801150,0.796462
체리,30,246.2010,8.206700,8.20750,5.747163,0.700301
마늘,29,2308.3710,79.599000,80.03700,33.385441,0.419420
물미역(청과),29,1565.5780,53.985448,62.93200,21.503372,0.398318
콩나물,29,1320.9590,45.550310,46.97060,16.903450,0.371094
숙주나물,29,1013.7635,34.957362,27.68450,27.192542,0.777877
파래(청과),29,349.2500,12.043103,13.68000,5.025101,0.417260
톳(청과),29,261.1780,9.006138,9.89200,4.154255,0.461269
망고,29,257.6910,8.885897,7.54800,6.065780,0.682630


In [28]:
CANDIDATE_ITEMS = [
    # 근채·양념채소 계열
    "마늘", "생강", "연근", "도라지",

    # 과일 계열
    "키위", "망고", "레몬", "자몽"
]

경매결과 json 생성

In [29]:
def fetch_auction_page(
    date,
    item,
    bubin="11000101",
    page=1,
    page_size=100
):
    params = {
        "id": "11116",
        "passwd": GARAK_AUCTION_PASSWORD,
        "dataid": "data12",
        "pagesize": page_size,
        "pageidx": page,
        "portal.templet": "false",
        "s_date": date,
        "s_bubin": bubin,
        "s_pummok": item,
        "s_sangi": "",
    }

    response = requests.get(
        BASE_URL,
        params=params,
        timeout=30
    )

    response.raise_for_status()

    try:
        data = response.json()
    except Exception:
        return []

    return data.get("resultData", [])

In [30]:
def fetch_all_auction_for_item(
    date,
    item,
    bubin="11000101",
    page_size=100,
    sleep_sec=0.15
):
    all_records = []

    for page in range(1, 1000):

        records = fetch_auction_page(
            date=date,
            item=item,
            bubin=bubin,
            page=page,
            page_size=page_size
        )

        if not records:
            break

        exact_records = [
            r for r in records
            if r.get("PUMMOK") == item
        ]

        all_records.extend(exact_records)

        # 페이지 종료 여부는 원래 API 응답 기준
        if len(records) < page_size:
            break

        time.sleep(sleep_sec)

    return all_records

In [31]:
test = fetch_all_auction_for_item(
    date="20220103",
    item="마늘"
)

print("마늘 거래건수:", len(test))

pd.DataFrame(test).head(20)

마늘 거래건수: 16


,ADJ_DT,DDD,PPRICE,PUM_NAME_IMSI,CORP_NM,UUN,ROWNO,PUMMOK,QTY,PUMJONG,INJUNG_GUBUN,SSANGI
0,20220103,특(1등),170000,[마늘]깐마늘(남도),서울청과,20kg,1,마늘,4,깐마늘(남도),일반,경남 창녕군
1,20220103,특(1등),155000,[마늘]깐마늘(남도),서울청과,20kg,2,마늘,2,깐마늘(남도),일반,경남 창녕군
2,20220103,특(1등),3500,[마늘]깐마늘(대서),서울청과,.3kg,3,마늘,228,깐마늘(대서),일반,경남 남해군
3,20220103,특(1등),2300,[마늘]깐마늘(대서),서울청과,.5kg,4,마늘,2497,깐마늘(대서),일반,경남 남해군
4,20220103,특(1등),65000,[마늘]깐마늘(대서),서울청과,20kg,5,마늘,10,깐마늘(대서),일반,경남 창녕군
5,20220103,특(1등),5450,[마늘]깐마늘(대서),서울청과,.5kg,6,마늘,80,깐마늘(대서),일반,경남 창녕군
6,20220103,특(1등),5450,[마늘]깐마늘(대서),서울청과,.5kg,7,마늘,280,깐마늘(대서),일반,경남 창녕군
7,20220103,특(1등),5450,[마늘]깐마늘(대서),서울청과,.5kg,8,마늘,280,깐마늘(대서),일반,경남 창녕군
8,20220103,특(1등),5450,[마늘]깐마늘(대서),서울청과,.5kg,9,마늘,80,깐마늘(대서),일반,경남 창녕군
9,20220103,특(1등),3500,[마늘]깐마늘(대서),서울청과,.3kg,10,마늘,228,깐마늘(대서),일반,경남 창녕군


In [32]:
dates = make_date_list(
    "20220101",
    "20220131"
)

auction_records = []

for date in dates:

    print(f"\n[{date}]")

    for item in CANDIDATE_ITEMS:

        records = fetch_all_auction_for_item(
            date=date,
            item=item,
            bubin="11000101"
        )

        for r in records:
            r["_query_date"] = date
            r["_query_item"] = item

        auction_records.extend(records)

        print(
            f"{item}: {len(records)}건",
            end=" | "
        )

    time.sleep(0.2)


[20220101]
마늘: 0건 | 생강: 0건 | 연근: 0건 | 도라지: 0건 | 키위: 0건 | 망고: 0건 | 레몬: 0건 | 자몽: 0건 | 
[20220102]
마늘: 0건 | 생강: 0건 | 연근: 0건 | 도라지: 0건 | 키위: 0건 | 망고: 0건 | 레몬: 0건 | 자몽: 0건 | 
[20220103]
마늘: 16건 | 생강: 0건 | 연근: 0건 | 도라지: 0건 | 키위: 2건 | 망고: 4건 | 레몬: 6건 | 자몽: 1건 | 
[20220104]
마늘: 5건 | 생강: 0건 | 연근: 0건 | 도라지: 0건 | 키위: 3건 | 망고: 2건 | 레몬: 7건 | 자몽: 0건 | 
[20220105]
마늘: 12건 | 생강: 0건 | 연근: 0건 | 도라지: 0건 | 키위: 16건 | 망고: 1건 | 레몬: 4건 | 자몽: 1건 | 
[20220106]
마늘: 8건 | 생강: 0건 | 연근: 0건 | 도라지: 0건 | 키위: 5건 | 망고: 5건 | 레몬: 3건 | 자몽: 1건 | 
[20220107]
마늘: 8건 | 생강: 0건 | 연근: 0건 | 도라지: 0건 | 키위: 15건 | 망고: 0건 | 레몬: 2건 | 자몽: 0건 | 
[20220108]
마늘: 0건 | 생강: 0건 | 연근: 0건 | 도라지: 0건 | 키위: 10건 | 망고: 5건 | 레몬: 1건 | 자몽: 1건 | 
[20220109]
마늘: 0건 | 생강: 0건 | 연근: 0건 | 도라지: 0건 | 키위: 0건 | 망고: 0건 | 레몬: 0건 | 자몽: 0건 | 
[20220110]
마늘: 8건 | 생강: 0건 | 연근: 0건 | 도라지: 0건 | 키위: 19건 | 망고: 3건 | 레몬: 5건 | 자몽: 0건 | 
[20220111]
마늘: 5건 | 생강: 0건 | 연근: 0건 | 도라지: 0건 | 키위: 0건 | 망고: 3건 | 레몬: 3건 | 자몽: 3건 | 
[20220112]
마늘: 13건 | 생강: 0건 | 연근: 0건 | 도라지: 0건 | 키위: 23건 |

In [33]:
auction_candidate = pd.DataFrame(auction_records)

print("전체 거래건수:", len(auction_candidate))

display(
    auction_candidate.head()
)

전체 거래건수: 596


,ADJ_DT,DDD,PPRICE,PUM_NAME_IMSI,CORP_NM,UUN,ROWNO,PUMMOK,QTY,PUMJONG,INJUNG_GUBUN,SSANGI,_query_date,_query_item
0,20220103,특(1등),170000,[마늘]깐마늘(남도),서울청과,20kg,1,마늘,4,깐마늘(남도),일반,경남 창녕군,20220103,마늘
1,20220103,특(1등),155000,[마늘]깐마늘(남도),서울청과,20kg,2,마늘,2,깐마늘(남도),일반,경남 창녕군,20220103,마늘
2,20220103,특(1등),3500,[마늘]깐마늘(대서),서울청과,.3kg,3,마늘,228,깐마늘(대서),일반,경남 남해군,20220103,마늘
3,20220103,특(1등),2300,[마늘]깐마늘(대서),서울청과,.5kg,4,마늘,2497,깐마늘(대서),일반,경남 남해군,20220103,마늘
4,20220103,특(1등),65000,[마늘]깐마늘(대서),서울청과,20kg,5,마늘,10,깐마늘(대서),일반,경남 창녕군,20220103,마늘


In [34]:
auction_candidate["PPRICE"] = pd.to_numeric(
    auction_candidate["PPRICE"],
    errors="coerce"
)

auction_candidate["QTY"] = pd.to_numeric(
    auction_candidate["QTY"],
    errors="coerce"
)

auction_candidate["ADJ_DT"] = pd.to_datetime(
    auction_candidate["ADJ_DT"],
    errors="coerce"
)

In [35]:
auction_summary = (
    auction_candidate
    .groupby("PUMMOK")
    .agg(
        n_trades=("PUMMOK", "size"),
        active_days=("ADJ_DT", "nunique"),
        n_varieties=("PUMJONG", "nunique"),
        n_origins=("SSANGI", "nunique"),
        median_price=("PPRICE", "median"),
        mean_price=("PPRICE", "mean"),
        std_price=("PPRICE", "std"),
        median_qty=("QTY", "median"),
        mean_qty=("QTY", "mean"),
    )
)

auction_summary["price_cv"] = (
    auction_summary["std_price"]
    / auction_summary["mean_price"]
)

auction_summary = auction_summary.sort_values(
    ["active_days", "n_trades"],
    ascending=False
)

display(auction_summary)

,n_trades,active_days,n_varieties,n_origins,median_price,mean_price,std_price,median_qty,mean_qty,price_cv
PUMMOK,,,,,,,,,,
레몬,101,24,2,4,20000.0,33258.415842,33468.720523,24.0,50.742574,1.006323
키위,222,23,3,8,28000.0,29988.288288,13844.346339,10.0,16.657658,0.461658
마늘,171,23,3,3,30500.0,66558.479532,69398.379800,36.0,282.040936,1.042668
망고,79,22,1,5,42000.0,40772.151899,14515.199255,10.0,28.202532,0.356008
자몽,23,13,1,2,42000.0,38739.130435,9076.530130,30.0,35.217391,0.234299


In [36]:
unit_check = (
    auction_candidate
    .groupby(["PUMMOK", "UUN"])
    .agg(
        n_trades=("PUMMOK", "size"),
        median_price=("PPRICE", "median"),
        median_qty=("QTY", "median")
    )
    .sort_values(
        ["PUMMOK", "n_trades"],
        ascending=[True, False]
    )
)

display(unit_check)

n_trades  median_price  median_qty
PUMMOK UUN                                       
레몬     17kg          49       65000.0        13.0
       .4kg          40        4500.0        24.0
       .5kg           6        4500.0        72.0
       4.5kg          3       38000.0         5.0
       5kg            3       26200.0       128.0
마늘     20kg          76      150000.0         6.0
       .5kg          27        5450.0       240.0
       .2kg          25        2300.0      1498.0
       .3kg          23        3500.0        84.0
       8kg           20       30500.0        10.0
망고     10kg          33       42000.0         1.0
       4kg           25       30000.0        30.0
       5kg           21       58000.0        20.0
자몽     16kg          16       42000.0        31.5
       15kg           4       41500.0        10.0
       13kg           1       39000.0        10.0
       4kg            1       20000.0        50.0
       5kg            1       20000.0        30.0
키위     10kg         176       33000.0         9.5
       6kg           13       20000.0        40.0
       5.6kg          6       20000.0        16.0
       5kg            6       14500.0        24.0
       4kg            5       15500.0        24.0
       3kg            3       15000.0       102.0
       5.8kg          3       20000.0        42.0
       11.5kg         2       72500.0        20.0
       2.31kg         2       15000.0        50.0
       2.3kg          2       14500.0        50.0
       3.5kg          2       15250.0       151.0
       11kg           1       48000.0        80.0
       8kg            1       53600.0         1.0

In [37]:
missing_items = sorted(
    set(CANDIDATE_ITEMS)
    - set(auction_candidate["PUMMOK"].dropna().unique())
)

print("경매결과가 없는 후보:", missing_items)

경매결과가 없는 후보: ['도라지', '생강', '연근']


In [40]:
import re
import numpy as np

def parse_unit_kg(x):
    if pd.isna(x):
        return np.nan

    x = str(x).lower().strip()

    match = re.search(r"(\d*\.?\d+)\s*kg", x)

    if match:
        return float(match.group(1))

    return np.nan


auction_candidate["unit_kg"] = (
    auction_candidate["UUN"]
    .apply(parse_unit_kg)
)

auction_candidate[
    ["PUMMOK", "UUN", "unit_kg"]
].drop_duplicates().sort_values(
    ["PUMMOK", "unit_kg"]
)

,PUMMOK,UUN,unit_kg
22,레몬,.4kg,0.40
39,레몬,.5kg,0.50
24,레몬,4.5kg,4.50
178,레몬,5kg,5.00
25,레몬,17kg,17.00
10,마늘,.2kg,0.20
2,마늘,.3kg,0.30
3,마늘,.5kg,0.50
32,마늘,8kg,8.00
0,마늘,20kg,20.00


In [42]:
auction_candidate["trade_qty_kg"] = (
    auction_candidate["QTY"]
    * auction_candidate["unit_kg"]
)

auction_candidate["price_per_kg"] = (
    auction_candidate["PPRICE"]
    / auction_candidate["unit_kg"]
)

In [43]:
auction_std = auction_candidate[
    (auction_candidate["unit_kg"] > 0) &
    (auction_candidate["QTY"] > 0) &
    (auction_candidate["PPRICE"] > 0)
].copy()

print(
    "표준화 가능률:",
    len(auction_std) / len(auction_candidate)
)

표준화 가능률: 1.0


In [44]:
std_summary = (
    auction_std
    .groupby("PUMMOK")
    .agg(
        n_trades=("PUMMOK", "size"),
        active_days=("ADJ_DT", "nunique"),

        median_price_kg=("price_per_kg", "median"),
        mean_price_kg=("price_per_kg", "mean"),
        std_price_kg=("price_per_kg", "std"),

        median_trade_qty_kg=("trade_qty_kg", "median"),
        mean_trade_qty_kg=("trade_qty_kg", "mean"),

        q25_trade_qty_kg=("trade_qty_kg",
                          lambda x: x.quantile(.25)),
        q75_trade_qty_kg=("trade_qty_kg",
                          lambda x: x.quantile(.75)),
    )
)

std_summary["price_cv"] = (
    std_summary["std_price_kg"]
    / std_summary["mean_price_kg"]
)

display(
    std_summary.sort_values(
        "n_trades",
        ascending=False
    )
)

,n_trades,active_days,median_price_kg,mean_price_kg,std_price_kg,median_trade_qty_kg,mean_trade_qty_kg,q25_trade_qty_kg,q75_trade_qty_kg,price_cv
PUMMOK,,,,,,,,,,
키위,222,23,3448.275862,3429.286763,1543.483877,100.0,126.077477,50.0,160.0,0.450089
마늘,171,23,8250.000000,8438.718324,3007.249502,96.0,178.143275,40.0,200.0,0.356363
레몬,101,24,8235.294118,7167.335792,3897.392625,57.6,229.485149,19.2,255.0,0.543771
망고,79,22,4800.000000,6749.367089,3387.358604,40.0,121.974684,10.0,100.0,0.501878
자몽,23,13,2666.666667,2758.333333,724.974965,320.0,517.739130,150.0,568.0,0.262831


In [45]:
unit_share = (
    auction_candidate
    .groupby(["PUMMOK", "UUN"])
    .size()
    .rename("n")
    .reset_index()
)

unit_share["share"] = (
    unit_share["n"]
    / unit_share.groupby("PUMMOK")["n"].transform("sum")
)

display(
    unit_share.sort_values(
        ["PUMMOK", "share"],
        ascending=[True, False]
    )
)

,PUMMOK,UUN,n,share
2,레몬,17kg,49,0.485149
0,레몬,.4kg,40,0.396040
1,레몬,.5kg,6,0.059406
3,레몬,4.5kg,3,0.029703
4,레몬,5kg,3,0.029703
8,마늘,20kg,76,0.444444
7,마늘,.5kg,27,0.157895
5,마늘,.2kg,25,0.146199
6,마늘,.3kg,23,0.134503
9,마늘,8kg,20,0.116959


In [46]:
unit_price_check = (
    auction_std
    .groupby(["PUMMOK", "UUN"])
    .agg(
        n=("price_per_kg", "size"),
        median_price_kg=("price_per_kg", "median"),
        mean_price_kg=("price_per_kg", "mean"),
        median_qty_kg=("trade_qty_kg", "median")
    )
    .sort_values(
        ["PUMMOK", "n"],
        ascending=[True, False]
    )
)

display(unit_price_check)

n  median_price_kg  mean_price_kg  median_qty_kg
PUMMOK UUN                                                       
레몬     17kg     49      3823.529412    3554.621849          221.0
       .4kg     40     11250.000000   11387.500000            9.6
       .5kg      6      9000.000000    9000.000000           36.0
       4.5kg     3      8444.444444    8148.148148           22.5
       5kg       3      5240.000000    5260.000000          640.0
마늘     20kg     76      7500.000000    6914.473684          120.0
       .5kg     27     10900.000000   10666.666667          120.0
       .2kg     25     11500.000000   11500.000000          299.6
       .3kg     23     11666.666667   11666.666667           25.2
       8kg      20      3812.500000    3684.375000           80.0
망고     10kg     33      4200.000000    4481.818182           10.0
       4kg      25      7500.000000    7380.000000          120.0
       5kg      21     11600.000000    9561.904762          100.0
자몽     16kg     16      2625.000000    2523.437500          504.0
       15kg      4      2766.666667    2766.666667          150.0
       13kg      1      3000.000000    3000.000000          130.0
       4kg       1      5000.000000    5000.000000          200.0
       5kg       1      4000.000000    4000.000000          150.0
키위     10kg    176      3300.000000    3158.522727           95.0
       6kg      13      3333.333333    3794.871795          240.0
       5.6kg     6      3571.428571    3422.619048           89.6
       5kg       6      2900.000000    3966.666667          120.0
       4kg       5      3875.000000    4265.000000           96.0
       3kg       3      5000.000000    4888.888889          306.0
       5.8kg     3      3448.275862    5919.540230          243.6
       11.5kg    2      6304.347826    6304.347826          230.0
       2.31kg    2      6493.506494    6493.506494          115.5
       2.3kg     2      6304.347826    6304.347826          115.0
       3.5kg     2      4357.142857    4357.142857          528.5
       11kg      1      4363.636364    4363.636364          880.0
       8kg       1      6700.000000    6700.000000            8.0

In [47]:
robust_summary = (
    auction_std
    .groupby("PUMMOK")["price_per_kg"]
    .agg(
        q10=lambda x: x.quantile(0.10),
        q25=lambda x: x.quantile(0.25),
        median="median",
        q75=lambda x: x.quantile(0.75),
        q90=lambda x: x.quantile(0.90),
    )
)

robust_summary["iqr"] = (
    robust_summary["q75"]
    - robust_summary["q25"]
)

robust_summary["iqr_ratio"] = (
    robust_summary["iqr"]
    / robust_summary["median"]
)

display(robust_summary)

,q10,q25,median,q75,q90,iqr,iqr_ratio
PUMMOK,,,,,,,
레몬,3352.941176,3823.529412,8235.294118,11250.000000,11750.000000,7426.470588,0.901786
마늘,3250.000000,7500.000000,8250.000000,11500.000000,11666.666667,4000.000000,0.484848
망고,4000.000000,4200.000000,4800.000000,10250.000000,12000.000000,6050.000000,1.260417
자몽,2475.000000,2625.000000,2666.666667,2902.083333,2987.500000,277.083333,0.103906
키위,1400.000000,2600.000000,3448.275862,4200.000000,4790.000000,1600.000000,0.464000


품목별 대표 거래규격을 정해서 가격 calibration에 사용

In [48]:
FRUIT_CANDIDATES = [
    "키위", "레몬", "망고", "자몽",
    "체리", "아보카도", "블루베리", "석류"
]

In [49]:
dates = make_date_list(
    "20220101",
    "20220131"
)

fruit_auction_records = []

for date in dates:

    print(f"\n[{date}]")

    for item in FRUIT_CANDIDATES:

        records = fetch_all_auction_for_item(
            date=date,
            item=item,
            bubin="11000101"   # 서울청과
        )

        for r in records:
            r["_query_date"] = date
            r["_query_item"] = item

        fruit_auction_records.extend(records)

        print(
            f"{item}: {len(records)}건",
            end=" | "
        )

    time.sleep(0.2)

fruit_auction = pd.DataFrame(fruit_auction_records)

print("\n\n전체 거래건수:", len(fruit_auction))
display(fruit_auction.head())


[20220101]
키위: 0건 | 레몬: 0건 | 망고: 0건 | 자몽: 0건 | 체리: 0건 | 아보카도: 0건 | 블루베리: 0건 | 석류: 0건 | 
[20220102]
키위: 0건 | 레몬: 0건 | 망고: 0건 | 자몽: 0건 | 체리: 0건 | 아보카도: 0건 | 블루베리: 0건 | 석류: 0건 | 
[20220103]
키위: 2건 | 레몬: 6건 | 망고: 4건 | 자몽: 1건 | 체리: 2건 | 아보카도: 0건 | 블루베리: 11건 | 석류: 1건 | 
[20220104]
키위: 3건 | 레몬: 7건 | 망고: 2건 | 자몽: 0건 | 체리: 0건 | 아보카도: 0건 | 블루베리: 3건 | 석류: 2건 | 
[20220105]
키위: 16건 | 레몬: 4건 | 망고: 1건 | 자몽: 1건 | 체리: 0건 | 아보카도: 0건 | 블루베리: 2건 | 석류: 4건 | 
[20220106]
키위: 5건 | 레몬: 3건 | 망고: 5건 | 자몽: 1건 | 체리: 2건 | 아보카도: 0건 | 블루베리: 5건 | 석류: 0건 | 
[20220107]
키위: 15건 | 레몬: 2건 | 망고: 0건 | 자몽: 0건 | 체리: 0건 | 아보카도: 0건 | 블루베리: 1건 | 석류: 1건 | 
[20220108]
키위: 10건 | 레몬: 1건 | 망고: 5건 | 자몽: 1건 | 체리: 0건 | 아보카도: 0건 | 블루베리: 8건 | 석류: 0건 | 
[20220109]
키위: 0건 | 레몬: 0건 | 망고: 0건 | 자몽: 0건 | 체리: 0건 | 아보카도: 0건 | 블루베리: 0건 | 석류: 0건 | 
[20220110]
키위: 19건 | 레몬: 5건 | 망고: 3건 | 자몽: 0건 | 체리: 0건 | 아보카도: 0건 | 블루베리: 2건 | 석류: 0건 | 
[20220111]
키위: 0건 | 레몬: 3건 | 망고: 3건 | 자몽: 3건 | 체리: 0건 | 아보카도: 0건 | 블루베리: 2건 | 석류: 2건 | 
[20220112]
키위: 23건 | 레몬: 4

,ADJ_DT,DDD,PPRICE,PUM_NAME_IMSI,CORP_NM,UUN,ROWNO,PUMMOK,QTY,PUMJONG,INJUNG_GUBUN,SSANGI,_query_date,_query_item
0,20220103,특(1등),15000,[키위]골드키위(2팀국산),서울청과,5.6kg,1,키위,19,골드키위(2팀국산),일반,제주 제주시,20220103,키위
1,20220103,특(1등),38000,[키위]키위(2팀국산),서울청과,10kg,2,키위,50,키위(2팀국산),일반,전남 보성군,20220103,키위
2,20220103,특(1등),4700,[레몬]레몬,서울청과,.4kg,1,레몬,16,레몬,일반,제주 제주시,20220103,레몬
3,20220103,특(1등),4500,[레몬]레몬,서울청과,.4kg,2,레몬,48,레몬,일반,제주 제주시,20220103,레몬
4,20220103,특(1등),38000,[레몬]레몬 (수입),서울청과,4.5kg,3,레몬,5,레몬 (수입),일반,수입산 멕시코,20220103,레몬


In [50]:
fruit_auction["PPRICE"] = pd.to_numeric(
    fruit_auction["PPRICE"],
    errors="coerce"
)

fruit_auction["QTY"] = pd.to_numeric(
    fruit_auction["QTY"],
    errors="coerce"
)

fruit_auction["ADJ_DT"] = pd.to_datetime(
    fruit_auction["ADJ_DT"],
    errors="coerce"
)

print("수집된 품목:")
print(sorted(fruit_auction["PUMMOK"].dropna().unique()))

수집된 품목:
['레몬', '망고', '블루베리', '석류', '자몽', '체리', '키위']


In [51]:
missing_fruits = sorted(
    set(FRUIT_CANDIDATES)
    - set(fruit_auction["PUMMOK"].dropna().unique())
)

print("경매결과가 없는 후보:", missing_fruits)

경매결과가 없는 후보: ['아보카도']


In [52]:
fruit_auction["unit_kg"] = (
    fruit_auction["UUN"]
    .apply(parse_unit_kg)
)

fruit_auction["trade_qty_kg"] = (
    fruit_auction["QTY"]
    * fruit_auction["unit_kg"]
)

fruit_auction["price_per_kg"] = (
    fruit_auction["PPRICE"]
    / fruit_auction["unit_kg"]
)

fruit_std = fruit_auction[
    (fruit_auction["unit_kg"].notna()) &
    (fruit_auction["unit_kg"] > 0) &
    (fruit_auction["QTY"] > 0) &
    (fruit_auction["PPRICE"] > 0)
].copy()

print("전체 거래건수:", len(fruit_auction))
print("kg 표준화 가능 거래건수:", len(fruit_std))
print(
    "kg 표준화 가능률:",
    round(len(fruit_std) / len(fruit_auction), 3)
)

전체 거래건수: 563
kg 표준화 가능 거래건수: 563
kg 표준화 가능률: 1.0


In [53]:
fruit_summary = (
    fruit_std
    .groupby("PUMMOK")
    .agg(
        n_trades=("PUMMOK", "size"),
        active_days=("ADJ_DT", "nunique"),
        n_varieties=("PUMJONG", "nunique"),
        n_origins=("SSANGI", "nunique"),

        median_price_kg=("price_per_kg", "median"),
        mean_price_kg=("price_per_kg", "mean"),
        std_price_kg=("price_per_kg", "std"),

        median_trade_qty_kg=("trade_qty_kg", "median"),
        mean_trade_qty_kg=("trade_qty_kg", "mean"),
    )
)

fruit_summary["price_cv_all"] = (
    fruit_summary["std_price_kg"]
    / fruit_summary["mean_price_kg"]
)

display(
    fruit_summary.sort_values(
        ["active_days", "n_trades"],
        ascending=False
    )
)

,n_trades,active_days,n_varieties,n_origins,median_price_kg,mean_price_kg,std_price_kg,median_trade_qty_kg,mean_trade_qty_kg,price_cv_all
PUMMOK,,,,,,,,,,
블루베리,91,25,2,4,6500.000000,31949.935650,51803.076097,10.0,55.286813,1.621383
레몬,101,24,2,4,8235.294118,7167.335792,3897.392625,57.6,229.485149,0.543771
키위,222,23,3,8,3448.275862,3429.286763,1543.483877,100.0,126.077477,0.450089
망고,79,22,1,5,4800.000000,6749.367089,3387.358604,40.0,121.974684,0.501878
석류,36,15,1,2,2200.000000,2880.555556,1882.979318,267.5,378.055556,0.653686
자몽,23,13,1,2,2666.666667,2758.333333,724.974965,320.0,517.739130,0.262831
체리,11,8,1,1,11800.000000,11981.818182,2969.787259,120.0,234.545455,0.247858


In [54]:
unit_count = (
    fruit_std
    .groupby(["PUMMOK", "UUN"])
    .size()
    .rename("n")
    .reset_index()
)

unit_count["share"] = (
    unit_count["n"]
    / unit_count.groupby("PUMMOK")["n"].transform("sum")
)

unit_count = unit_count.sort_values(
    ["PUMMOK", "n"],
    ascending=[True, False]
)

display(unit_count)

,PUMMOK,UUN,n,share
2,레몬,17kg,49,0.485149
0,레몬,.4kg,40,0.396040
1,레몬,.5kg,6,0.059406
3,레몬,4.5kg,3,0.029703
4,레몬,5kg,3,0.029703
5,망고,10kg,33,0.417722
6,망고,4kg,25,0.316456
7,망고,5kg,21,0.265823
11,블루베리,10kg,51,0.560440
10,블루베리,1.5kg,18,0.197802


In [55]:
representative_units = (
    unit_count
    .sort_values(
        ["PUMMOK", "n"],
        ascending=[True, False]
    )
    .groupby("PUMMOK")
    .first()
    .reset_index()
    .rename(columns={
        "UUN": "representative_unit",
        "n": "representative_n",
        "share": "representative_share"
    })
)

display(representative_units)

,PUMMOK,representative_unit,representative_n,representative_share
0,레몬,17kg,49,0.485149
1,망고,10kg,33,0.417722
2,블루베리,10kg,51,0.560440
3,석류,5kg,36,1.000000
4,자몽,16kg,16,0.695652
5,체리,5kg,11,1.000000
6,키위,10kg,176,0.792793


In [56]:
fruit_rep = fruit_std.merge(
    representative_units[
        ["PUMMOK", "representative_unit"]
    ],
    on="PUMMOK",
    how="left"
)

fruit_rep = fruit_rep[
    fruit_rep["UUN"]
    == fruit_rep["representative_unit"]
].copy()

display(
    fruit_rep[
        [
            "PUMMOK",
            "UUN",
            "PPRICE",
            "QTY",
            "price_per_kg",
            "trade_qty_kg"
        ]
    ].head(20)
)

,PUMMOK,UUN,PPRICE,QTY,price_per_kg,trade_qty_kg
1,키위,10kg,38000,50,3800.000000,500.0
5,레몬,17kg,63000,54,3705.882353,918.0
6,레몬,17kg,60000,27,3529.411765,459.0
7,레몬,17kg,60000,27,3529.411765,459.0
8,망고,10kg,48000,2,4800.000000,20.0
9,망고,10kg,42000,1,4200.000000,10.0
10,망고,10kg,42000,1,4200.000000,10.0
11,망고,10kg,42000,1,4200.000000,10.0
12,자몽,16kg,39000,63,2437.500000,1008.0
13,체리,5kg,95000,7,19000.000000,35.0


In [57]:
rep_summary = (
    fruit_rep
    .groupby("PUMMOK")
    .agg(
        rep_n_trades=("PUMMOK", "size"),
        rep_active_days=("ADJ_DT", "nunique"),

        rep_median_price_kg=("price_per_kg", "median"),
        rep_mean_price_kg=("price_per_kg", "mean"),
        rep_std_price_kg=("price_per_kg", "std"),

        rep_q25_price_kg=(
            "price_per_kg",
            lambda x: x.quantile(0.25)
        ),
        rep_q75_price_kg=(
            "price_per_kg",
            lambda x: x.quantile(0.75)
        ),

        rep_median_qty_kg=("trade_qty_kg", "median"),
        rep_mean_qty_kg=("trade_qty_kg", "mean"),
    )
)

rep_summary["rep_price_cv"] = (
    rep_summary["rep_std_price_kg"]
    / rep_summary["rep_mean_price_kg"]
)

rep_summary["rep_price_iqr_ratio"] = (
    (
        rep_summary["rep_q75_price_kg"]
        - rep_summary["rep_q25_price_kg"]
    )
    / rep_summary["rep_median_price_kg"]
)

display(rep_summary)

,rep_n_trades,rep_active_days,rep_median_price_kg,rep_mean_price_kg,rep_std_price_kg,rep_q25_price_kg,rep_q75_price_kg,rep_median_qty_kg,rep_mean_qty_kg,rep_price_cv,rep_price_iqr_ratio
PUMMOK,,,,,,,,,,,
레몬,49,15,3823.529412,3554.621849,1649.289456,3529.411765,3823.529412,221.0,400.714286,0.463985,0.076923
망고,33,16,4200.000000,4481.818182,381.980485,4200.000000,4800.000000,10.0,13.030303,0.085229,0.142857
블루베리,51,21,6500.000000,6301.960784,364.137444,6200.000000,6500.000000,10.0,33.333333,0.057782,0.046154
석류,36,15,2200.000000,2880.555556,1882.979318,1400.000000,4000.000000,267.5,378.055556,0.653686,1.181818
자몽,16,8,2625.000000,2523.437500,517.342468,2625.000000,2812.500000,504.0,628.000000,0.205015,0.071429
체리,11,8,11800.000000,11981.818182,2969.787259,10200.000000,12200.000000,120.0,234.545455,0.247858,0.169492
키위,176,18,3300.000000,3158.522727,1274.905522,2200.000000,4025.000000,95.0,110.340909,0.403640,0.553030


In [58]:
fruit_compare = (
    fruit_summary
    .join(
        rep_summary,
        how="left"
    )
    .reset_index()
    .merge(
        representative_units[
            [
                "PUMMOK",
                "representative_unit",
                "representative_n",
                "representative_share"
            ]
        ],
        on="PUMMOK",
        how="left"
    )
    .set_index("PUMMOK")
)

fruit_compare = fruit_compare[
    [
        "n_trades",
        "active_days",
        "n_varieties",
        "n_origins",

        "representative_unit",
        "representative_share",

        "rep_n_trades",
        "rep_active_days",

        "rep_median_price_kg",
        "rep_price_cv",
        "rep_price_iqr_ratio",

        "median_trade_qty_kg",
        "rep_median_qty_kg",
    ]
]

display(
    fruit_compare.sort_values(
        ["active_days", "n_trades"],
        ascending=False
    )
)

,n_trades,active_days,n_varieties,n_origins,representative_unit,representative_share,rep_n_trades,rep_active_days,rep_median_price_kg,rep_price_cv,rep_price_iqr_ratio,median_trade_qty_kg,rep_median_qty_kg
PUMMOK,,,,,,,,,,,,,
블루베리,91,25,2,4,10kg,0.560440,51,21,6500.000000,0.057782,0.046154,10.0,10.0
레몬,101,24,2,4,17kg,0.485149,49,15,3823.529412,0.463985,0.076923,57.6,221.0
키위,222,23,3,8,10kg,0.792793,176,18,3300.000000,0.403640,0.553030,100.0,95.0
망고,79,22,1,5,10kg,0.417722,33,16,4200.000000,0.085229,0.142857,40.0,10.0
석류,36,15,1,2,5kg,1.000000,36,15,2200.000000,0.653686,1.181818,267.5,267.5
자몽,23,13,1,2,16kg,0.695652,16,8,2625.000000,0.205015,0.071429,320.0,504.0
체리,11,8,1,1,5kg,1.000000,11,8,11800.000000,0.247858,0.169492,120.0,120.0


In [59]:
SELECTED_ITEMS = [
    "키위",
    "블루베리",
    "망고"
]

In [60]:
selected_auction = fruit_std[
    fruit_std["PUMMOK"].isin(SELECTED_ITEMS)
].copy()

selected_rep = fruit_rep[
    fruit_rep["PUMMOK"].isin(SELECTED_ITEMS)
].copy()

display(
    fruit_compare.loc[
        SELECTED_ITEMS
    ]
)

,n_trades,active_days,n_varieties,n_origins,representative_unit,representative_share,rep_n_trades,rep_active_days,rep_median_price_kg,rep_price_cv,rep_price_iqr_ratio,median_trade_qty_kg,rep_median_qty_kg
PUMMOK,,,,,,,,,,,,,
키위,222,23,3,8,10kg,0.792793,176,18,3300.0,0.403640,0.553030,100.0,95.0
블루베리,91,25,2,4,10kg,0.560440,51,21,6500.0,0.057782,0.046154,10.0,10.0
망고,79,22,1,5,10kg,0.417722,33,16,4200.0,0.085229,0.142857,40.0,10.0
